# Merging and Joining Data

**Learning Objectives:**
- Combine DataFrames vertically with `pd.concat()`
- Merge DataFrames horizontally with `pd.merge()` (inner, outer, left, right)
- Use the `join()` method for index-based merges
- Handle overlapping column names with `suffixes`
- Understand the deprecated `append()`

## 1. Setup – Sample DataFrames

We'll use customers, orders, and monthly reports.

In [ ]:
import pandas as pd
import numpy as np

# Customers
customers = pd.DataFrame({
    "CustomerID": [1, 2, 3, 4],
    "Name": ["Alice", "Bob", "Charlie", "Diana"],
    "City": ["NYC", "LA", "Chicago", "Houston"]
})

# Orders
orders = pd.DataFrame({
    "OrderID": [101, 102, 103, 104, 105],
    "CustomerID": [1, 2, 1, 3, 5],
    "Amount": [250, 450, 300, 700, 150],
    "Date": pd.date_range("2025-01-01", periods=5, freq="D")
})

print("Customers:")
print(customers)
print("\nOrders:")
print(orders)

## 2. `pd.concat()` – Vertical / Horizontal Stacking

Concatenates along an axis. Default (`axis=0`) stacks rows.

In [ ]:
# Monthly reports (same columns)
jan = pd.DataFrame({"Product": ["A", "B"], "Sales": [100, 200]})
feb = pd.DataFrame({"Product": ["A", "C"], "Sales": [150, 250]})
mar = pd.DataFrame({"Product": ["B", "D"], "Sales": [180, 300]})

# Vertical concatenation
q1 = pd.concat([jan, feb, mar], ignore_index=True)
print("Q1 combined:")
print(q1)

# Horizontal concatenation
df1 = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
df2 = pd.DataFrame({"C": [5, 6], "D": [7, 8]})
print("\nHorizontal concat (axis=1):")
print(pd.concat([df1, df2], axis=1))

## 3. `pd.merge()` – SQL-Style Joins

Merge two DataFrames on a key column. Supports `how='inner'|'outer'|'left'|'right'`.

In [ ]:
# Inner join – only matching CustomerID in both
inner = pd.merge(customers, orders, on="CustomerID", how="inner")
print("Inner join:\n", inner, sep="")

# Left join – all customers, orders where available
left = pd.merge(customers, orders, on="CustomerID", how="left")
print("\nLeft join:\n", left, sep="")

# Right join – all orders, customers where available
right = pd.merge(customers, orders, on="CustomerID", how="right")
print("\nRight join:\n", right, sep="")

# Outer join – all records from both sides
outer = pd.merge(customers, orders, on="CustomerID", how="outer")
print("\nOuter join:\n", outer, sep="")

## 4. Handling Overlapping Column Names

When merging DataFrames with **different key column names** or **same-named non-key columns**, use `left_on`, `right_on`, and `suffixes`.

In [ ]:
# Different key names
cust2 = customers.rename(columns={"CustomerID": "CID"})
merged_diff = pd.merge(cust2, orders, left_on="CID", right_on="CustomerID", how="inner")
print("Merge on different key names:")
print(merged_diff)

# Overlapping non-key columns (same column name in both)
df_a = pd.DataFrame({"key": [1, 2], "value": ["a1", "a2"], "data": [10, 20]})
df_b = pd.DataFrame({"key": [1, 2], "value": ["b1", "b2"], "info": [100, 200]})
merged_suffix = pd.merge(df_a, df_b, on="key", suffixes=("_left", "_right"))
print("\nWith suffixes:")
print(merged_suffix)

## 5. `join()` – Index-Based Merge

Convenience method to merge on the index.

In [ ]:
left_df = pd.DataFrame({"A": [1, 2, 3]}, index=["x", "y", "z"])
right_df = pd.DataFrame({"B": [4, 5, 6]}, index=["x", "y", "w"])

joined = left_df.join(right_df, how="outer")
print("join():")
print(joined)

## 6. `append()` – Deprecated

`df.append()` is **deprecated** since Pandas 1.4. Use `pd.concat()` instead.

In [ ]:
# Old way (deprecated – do not use):
# df1.append(df2)

# New way:
appended = pd.concat([jan, feb], ignore_index=True)
print(appended)

## 7. Practice Problems

In [ ]:
# Problem 1: Merge customers and orders to find the total amount
#            spent by each customer. Show Name and TotalAmount.
merged = pd.merge(customers, orders, on="CustomerID")
total_by_customer = merged.groupby("Name")["Amount"].sum().reset_index()
total_by_customer.columns = ["Name", "TotalAmount"]
print(total_by_customer)

In [ ]:
# Problem 2: Concatenate the 3 monthly DataFrames (jan, feb, mar)
#            and add a 'Month' column to track origin.
jan_m = jan.copy(); jan_m["Month"] = "Jan"
feb_m = feb.copy(); feb_m["Month"] = "Feb"
mar_m = mar.copy(); mar_m["Month"] = "Mar"

all_months = pd.concat([jan_m, feb_m, mar_m], ignore_index=True)
print(all_months)

In [ ]:
# Problem 3: Outer join customers and orders. Note the NaN values
#            for customers without orders and orders without customers.
outer = pd.merge(customers, orders, on="CustomerID", how="outer", indicator=True)
print(outer)

## 8. Summary

| Method | Use Case |
|--------|----------|
| `pd.concat()` | Stack DataFrames vertically or horizontally |
| `pd.merge()` | SQL-style joins on key columns |
| `df.join()` | Merge on index (convenience) |
| `suffixes` | Disambiguate overlapping column names |
| `indicator` | Add a column showing merge origin in `merge()` |